# Sidewalk hazard detector — training

Fine-tunes YOLO26n into the four-class detector the app expects, on a free Colab T4, and exports it as Core ML for iOS.

The model matches the one named in the investigation report. The export format does not: the report says TensorFlow Lite, and this exports Core ML, because the app is iOS-first and TFLite on iOS runs on the CPU or GPU rather than the Neural Engine. That is a deliberate change worth a sentence in the report rather than a silent one. The same trained weights export to TFLite as well if the Android path is needed later — one extra line, no retraining.

**Classes, and where the data comes from.** The order below is not arbitrary: it matches `HAZARD_CLASSES` in `src/types/hazard.ts` exactly, so the model's class indices are the app's class indices and nothing has to be remapped in Swift.

| # | Class | Source | Coverage |
|---|---|---|---|
| 0 | `pothole` | RDD2022, damage type D40 | Excellent — thousands of instances |
| 1 | `slippery-surface` | Roboflow puddle / wet-surface sets | **Weak** — a couple of thousand images at best |
| 2 | `broken-tactile-paving` | RDD2022, types D00/D10/D20 (cracks) | Good, but see the note below |
| 3 | `pathway-obstruction` | COCO subset, all collapsed to one class | Excellent — COCO is enormous |

**Two honest caveats, worth writing into the report rather than hiding.**

Class 2 is trained on road cracking, not on tactile paving specifically. That is defensible because the app's own setting is labelled *"Broken Tactile Paving / Uneven Surface"* — cracked and broken pavement is the hazard, and it is what the walker needs warning about. But a model trained this way will not reliably recognise *intact* tactile paving as distinct from a cracked slab. If tactile paving specifically matters, add a dedicated dataset (see the search terms in the tactile cell) or collect your own.

Class 1 is the weak one. There is no substantial public dataset of slippery surfaces with bounding boxes, because "slippery" is not visible — standing water is, and that is what these datasets actually label. Expect this class to underperform the other three, report it as a limitation, and consider collecting a few hundred of your own images if it matters to your marks.

**Free tier survival.** Sessions drop without warning. Everything is written to Drive, and the training cell is resume-safe — if you get disconnected, run the resume cell and it picks up from the last epoch rather than starting over.

## 1. Runtime

Set **Runtime → Change runtime type → T4 GPU** before running anything. If the check below says CUDA is unavailable, you are on a CPU runtime and training will take days rather than hours.

In [ ]:
!nvidia-smi

import torch
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
    print('vram:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# Weights and results live on Drive so a dropped session costs minutes, not hours.
PROJECT = Path('/content/drive/MyDrive/hazard-detector')
PROJECT.mkdir(parents=True, exist_ok=True)

# The dataset itself stays on local disk - it is large, and Drive I/O during
# training is slow enough to bottleneck the GPU.
DATA = Path('/content/data')
DATA.mkdir(parents=True, exist_ok=True)

print('project:', PROJECT)
print('data   :', DATA)

In [ ]:
# Deliberately short. Ultralytics brings almost everything; coremltools does the
# export; roboflow fetches the puddle sets.
#
# Note what is *not* here: fiftyone, the usual way to pull a COCO subset. It
# installs MongoDB and its own Pillow pin, and that pin collides with the one
# Colab ships - leaving PIL half-upgraded and every image library broken with
# "cannot import name '_Ink' from 'PIL._typing'". COCO is served over plain
# HTTP, so section 4 fetches it directly and the whole dependency goes away.
!pip -q install -U ultralytics coremltools roboflow

import ultralytics
ultralytics.checks()

## 2. Class definitions

Change these and you change the model's contract with the app. The indices must stay aligned with `HAZARD_CLASSES`.

In [ ]:
CLASSES = ['pothole', 'slippery-surface', 'broken-tactile-paving', 'pathway-obstruction']
POTHOLE, SLIPPERY, UNEVEN, OBSTRUCTION = range(4)

# Where each converted dataset gets written, one directory per source.
STAGE = DATA / 'stage'
STAGE.mkdir(parents=True, exist_ok=True)

import collections, random, shutil, os
random.seed(0)

def write_label(dst_dir, stem, rows):
    """rows: list of (cls, xc, yc, w, h) already normalised to 0-1."""
    with open(dst_dir / (stem + '.txt'), 'w') as fh:
        for cls, xc, yc, w, h in rows:
            fh.write(f'{cls} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n')

def stage_dirs(name):
    images = STAGE / name / 'images'
    labels = STAGE / name / 'labels'
    images.mkdir(parents=True, exist_ok=True)
    labels.mkdir(parents=True, exist_ok=True)
    return images, labels

## 3. RDD2022 — potholes and cracked surface

47,420 road images from six countries, annotated in Pascal VOC XML with damage types D00 (longitudinal crack), D10 (transverse crack), D20 (alligator crack) and D40 (pothole). Two of our four classes come from here.

The cell below lists the files from the figshare API rather than hard-coding a download URL, because figshare's per-file URLs change between versions. Pick the countries you want from the listing — **do not take all six on a free runtime**, it is roughly 12 GB and the download alone will eat your session. Japan plus Czech is a good balance of size and variety.

> Arya, D., Maeda, H., Kumar Ghosh, S., Toshniwal, D., & Sekimoto, Y. (2022). *RDD2022: A multi-national image dataset for automatic Road Damage Detection.* [doi:10.6084/m9.figshare.21431547](https://doi.org/10.6084/m9.figshare.21431547)

In [ ]:
import requests

meta = requests.get('https://api.figshare.com/v2/articles/21431547').json()
print(meta['title'], '\n')
for f in meta['files']:
    print(f"{f['name']:34s} {f['size'] / 1e9:6.2f} GB")

FILES = {f['name']: f['download_url'] for f in meta['files']}

In [ ]:
# Pick from the names printed above. Start small - you can always add a country
# and re-run the conversion.
WANTED = [name for name in FILES if 'Japan' in name or 'Czech' in name]
print('downloading:', WANTED)

RDD = DATA / 'rdd'
RDD.mkdir(parents=True, exist_ok=True)

for name in WANTED:
    target = RDD / name
    if target.exists():
        print('already have', name)
        continue
    !wget -q --show-progress -O "{target}" "{FILES[name]}"
    !unzip -q -o "{target}" -d "{RDD}"

!du -sh "{RDD}"

In [ ]:
import xml.etree.ElementTree as ET

# D40 is a pothole. The three crack types all describe a broken, uneven walking
# surface, which is the hazard the app's "Broken Tactile Paving / Uneven Surface"
# setting covers - so they collapse into one class rather than three the user
# would never distinguish between.
RDD_MAP = {'D40': POTHOLE, 'D00': UNEVEN, 'D10': UNEVEN, 'D20': UNEVEN}

# Cracks outnumber potholes heavily in RDD2022. Left alone the model learns to
# call everything a crack, so crack-only images are subsampled while every image
# containing a pothole is kept.
CRACK_ONLY_KEEP = 0.35

images_out, labels_out = stage_dirs('rdd')
counts = collections.Counter()
kept = skipped = 0

for xml_path in RDD.rglob('annotations/xmls/*.xml'):
    root = ET.parse(xml_path).getroot()
    size = root.find('size')
    W, H = int(size.find('width').text), int(size.find('height').text)
    if W == 0 or H == 0:
        continue

    rows = []
    for obj in root.findall('object'):
        name = obj.find('name').text.strip()
        if name not in RDD_MAP:
            continue
        box = obj.find('bndbox')
        x1, y1 = float(box.find('xmin').text), float(box.find('ymin').text)
        x2, y2 = float(box.find('xmax').text), float(box.find('ymax').text)
        x1, x2 = sorted((max(0, x1), min(W, x2)))
        y1, y2 = sorted((max(0, y1), min(H, y2)))
        if x2 - x1 < 2 or y2 - y1 < 2:
            continue
        rows.append((RDD_MAP[name], ((x1 + x2) / 2) / W, ((y1 + y2) / 2) / H,
                     (x2 - x1) / W, (y2 - y1) / H))

    if not rows:
        continue
    if POTHOLE not in {r[0] for r in rows} and random.random() > CRACK_ONLY_KEEP:
        skipped += 1
        continue

    stem = xml_path.stem
    src_img = xml_path.parent.parent.parent / 'images' / (stem + '.jpg')
    if not src_img.exists():
        continue

    shutil.copy(src_img, images_out / src_img.name)
    write_label(labels_out, stem, rows)
    counts.update(CLASSES[r[0]] for r in rows)
    kept += 1

print('images kept:', kept, ' crack-only images dropped:', skipped)
print('instances  :', dict(counts))

## 4. COCO subset — pathway obstruction

Nothing needs training from scratch here. COCO already contains, labelled and in quantity, exactly the things that block a pavement: people, bicycles, parked cars, bins, benches, dogs. They all collapse into one class, because the app does not care *what* is in the way — the guidance is the same either way.

`COCO_MAX_IMAGES` is the knob that keeps this class from swamping the other three. Four thousand is roughly the size of the RDD contribution.

Only the annotations are downloaded in bulk; the images are fetched individually, and only the ones actually chosen — about 650 MB rather than the full 19 GB of COCO train2017.

In [ ]:
import json, gc, urllib.request
from concurrent.futures import ThreadPoolExecutor

OBSTRUCTION_CLASSES = [
    'person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck',
    'bench', 'chair', 'potted plant', 'dog', 'suitcase', 'backpack',
    'fire hydrant', 'stop sign', 'parking meter',
]
COCO_MAX_IMAGES = 4000

COCO = DATA / 'coco'
COCO.mkdir(parents=True, exist_ok=True)
ann_zip = COCO / 'annotations_trainval2017.zip'
ann_json = COCO / 'annotations' / 'instances_train2017.json'

if not ann_json.exists():
    if not ann_zip.exists():
        !wget -q --show-progress -O "{ann_zip}" http://images.cocodataset.org/annotations/annotations_trainval2017.zip
    !unzip -q -o "{ann_zip}" 'annotations/instances_train2017.json' -d "{COCO}"

print('parsing annotations - this takes a minute and a few GB of RAM')
with open(ann_json) as fh:
    raw = json.load(fh)

In [ ]:
name_to_id = {c['name']: c['id'] for c in raw['categories']}
wanted = {name_to_id[n] for n in OBSTRUCTION_CLASSES if n in name_to_id}
missing = [n for n in OBSTRUCTION_CLASSES if n not in name_to_id]
if missing:
    print('not COCO categories, ignored:', missing)

# image id -> its boxes, keeping only the categories we asked for. Everything
# else in the image is dropped: these images are full of labelled sofas and
# sandwiches, and teaching those as pavement obstructions would be worse than
# not training the class at all.
boxes = collections.defaultdict(list)
for ann in raw['annotations']:
    if ann['category_id'] in wanted and not ann.get('iscrowd', 0):
        boxes[ann['image_id']].append(ann['bbox'])

meta = {im['id']: im for im in raw['images'] if im['id'] in boxes}

# The parsed JSON is several GB and is not needed again.
del raw
gc.collect()

chosen = sorted(meta)
random.Random(0).shuffle(chosen)
chosen = chosen[:COCO_MAX_IMAGES]
print(len(boxes), 'images contain a wanted class;', len(chosen), 'selected')

In [ ]:
images_out, labels_out = stage_dirs('coco')

def fetch(image_id):
    info = meta[image_id]
    W, H = info['width'], info['height']
    destination = images_out / info['file_name']

    rows = []
    for x, y, w, h in boxes[image_id]:
        if w / W < 0.02 or h / H < 0.02:   # specks the detector cannot use
            continue
        rows.append((OBSTRUCTION, (x + w / 2) / W, (y + h / 2) / H, w / W, h / H))
    if not rows:
        return 0

    if not destination.exists():
        try:
            urllib.request.urlretrieve(info['coco_url'], destination)
        except Exception:
            return 0

    write_label(labels_out, destination.stem, rows)
    return 1

# Threaded because this is entirely network-bound - sequentially it would take
# well over an hour.
with ThreadPoolExecutor(max_workers=16) as pool:
    kept = sum(pool.map(fetch, chosen))

print('coco images staged:', kept)

## 5. Puddles and wet surface — slippery

The weak class. These are small community datasets rather than benchmarks, so quality varies and you should look at a few images before trusting them.

Get a free key from [roboflow.com](https://roboflow.com) → Settings → API keys. Candidates worth pulling, largest first:

- `case-ows0c/water-and-wet-surface` — ~1,800 images
- `icews/water-puddles-f5tdo` — ~400 images
- `puddle-water-detection/pkl-hkosw` — ~250 images

Search [universe.roboflow.com](https://universe.roboflow.com/search?q=class%3Apuddle) for more. If you later collect your own wet-pavement photos, they go in the same staging directory and need no other change.

In [ ]:
ROBOFLOW_KEY = ''   # paste your key

from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_KEY)

# (workspace, project) pairs, taken from the universe URL:
#   universe.roboflow.com/<workspace>/<project>
SOURCES = [
    ('case-ows0c', 'water-and-wet-surface'),
    ('icews', 'water-puddles-f5tdo'),
]

downloaded = []
for workspace, project_id in SOURCES:
    try:
        project = rf.workspace(workspace).project(project_id)
        version = max(v.version for v in project.versions())
        out = DATA / 'wet' / project_id
        project.version(version).download('yolov8', location=str(out))
        downloaded.append(out)
        print('ok:', project_id, 'v' + str(version))
    except Exception as exc:
        print('skipped', project_id, '-', exc)

print(downloaded)

In [ ]:
# Tolerates the Roboflow cell having been skipped entirely - the other three
# classes still train, this one is simply absent.
downloaded = globals().get('downloaded', [])

images_out, labels_out = stage_dirs('wet')
kept = 0

# Every class in these datasets - "puddle", "water", "wet surface" - means the
# same thing to the app, so all of them collapse to one index.
for root in downloaded:
    for label_file in root.rglob('labels/*.txt'):
        image_file = None
        for ext in ('.jpg', '.jpeg', '.png'):
            candidate = label_file.parent.parent / 'images' / (label_file.stem + ext)
            if candidate.exists():
                image_file = candidate
                break
        if image_file is None:
            continue

        rows = []
        for line in label_file.read_text().strip().splitlines():
            parts = line.split()
            if len(parts) < 5:
                continue
            rows.append((SLIPPERY, *[float(p) for p in parts[1:5]]))
        if not rows:
            continue

        # Names collide across datasets, so they are prefixed.
        stem = root.name + '_' + label_file.stem
        shutil.copy(image_file, images_out / (stem + image_file.suffix))
        write_label(labels_out, stem, rows)
        kept += 1

print('wet-surface images staged:', kept)

## 6. Merge, split, and look at the balance

The split is by image, 80/20, with a fixed seed so a re-run gives the same split — otherwise validation scores drift between runs for reasons that have nothing to do with the model.

Read the instance counts carefully. If one class has an order of magnitude more instances than another, the model will quietly learn to favour it, and mAP averaged over classes will hide that.

In [ ]:
import yaml

DATASET = DATA / 'hazards'
for split in ('train', 'val'):
    (DATASET / split / 'images').mkdir(parents=True, exist_ok=True)
    (DATASET / split / 'labels').mkdir(parents=True, exist_ok=True)

pairs = []
for stage in STAGE.iterdir():
    if not (stage / 'images').is_dir():
        continue
    for image in (stage / 'images').iterdir():
        label = stage / 'labels' / (image.stem + '.txt')
        if label.exists():
            pairs.append((image, label))

random.Random(0).shuffle(pairs)
cut = int(len(pairs) * 0.8)

for split, chunk in (('train', pairs[:cut]), ('val', pairs[cut:])):
    for image, label in chunk:
        shutil.copy(image, DATASET / split / 'images' / image.name)
        shutil.copy(label, DATASET / split / 'labels' / label.name)

data_yaml = DATASET / 'hazards.yaml'
data_yaml.write_text(yaml.safe_dump({
    'path': str(DATASET),
    'train': 'train/images',
    'val': 'val/images',
    'names': {i: name for i, name in enumerate(CLASSES)},
}, sort_keys=False))

print(data_yaml.read_text())
print('train:', cut, ' val:', len(pairs) - cut)

In [ ]:
for split in ('train', 'val'):
    per_class = collections.Counter()
    images_with = collections.Counter()
    for label in (DATASET / split / 'labels').iterdir():
        present = set()
        for line in label.read_text().strip().splitlines():
            if line:
                cls = int(line.split()[0])
                per_class[CLASSES[cls]] += 1
                present.add(CLASSES[cls])
        images_with.update(present)

    print(f'--- {split} ---')
    for name in CLASSES:
        print(f'  {name:24s} {per_class[name]:6d} instances in {images_with[name]:5d} images')

## 7. Train

YOLO26n — the nano variant, and the model the investigation report already names. On a phone that is the only sensible size: it runs in a few milliseconds on the Neural Engine, where a small or medium model would drain the battery and drop frames while barely improving on hazards this large in frame.

YOLO26 is end-to-end by default, meaning its detection head emits final boxes with no non-maximum suppression step at all. That matters more on a phone than the accuracy figures do: NMS is post-processing that runs on the CPU every frame, and removing it takes work off the same core that is holding the AR tracking together.

Roughly 1.5–2 minutes per epoch on a T4 at this dataset size, so 80 epochs is around two to three hours. That is longer than a free session is guaranteed to last, which is what the resume cell below is for.

If you get disconnected: re-run the setup cells (runtime, Drive, install), **skip the dataset cells**, and run the resume cell instead of this one.

In [ ]:
from ultralytics import YOLO

RUN_NAME = 'hazard-yolo26n'

model = YOLO('yolo26n.pt')          # COCO-pretrained; training from scratch on
                                    # this little data would be far worse
results = model.train(
    data=str(data_yaml),
    epochs=80,
    imgsz=640,
    batch=16,                       # fits a T4 comfortably at 640
    workers=2,                      # Colab throttles above this
    patience=20,                    # stop if val stops improving
    close_mosaic=10,                # last 10 epochs without mosaic, so the model
                                    # finishes on images shaped like real frames
    project=str(PROJECT / 'runs'),
    name=RUN_NAME,
    exist_ok=True,
    seed=0,
    plots=True,
)

In [ ]:
# RESUME ONLY - run this instead of the cell above after a dropped session.
from ultralytics import YOLO

model = YOLO(str(PROJECT / 'runs' / RUN_NAME / 'weights' / 'last.pt'))
results = model.train(resume=True)

## 8. Validate

Look at per-class numbers, not the headline mAP. A strong average across four classes can hide one class being close to useless — which, given where the data came from, is the expected failure mode for `slippery-surface`.

The confusion matrix written into the run directory is worth putting in the report.

In [ ]:
best = YOLO(str(PROJECT / 'runs' / RUN_NAME / 'weights' / 'best.pt'))
metrics = best.val(data=str(data_yaml), imgsz=640, plots=True)

print(f"{'class':24s} {'P':>7s} {'R':>7s} {'mAP50':>7s} {'mAP50-95':>9s}")
for i, name in enumerate(CLASSES):
    p, r, map50, map5095 = metrics.class_result(i)
    print(f'{name:24s} {p:7.3f} {r:7.3f} {map50:7.3f} {map5095:9.3f}')
print(f"\n{'overall':24s} {'':7s} {'':7s} {metrics.box.map50:7.3f} {metrics.box.map:9.3f}")

## 9. Export to Core ML

The two export arguments below need explaining together, because they look like they contradict the point of YOLO26.

YOLO26's end-to-end head emits a raw `(1, 300, 6)` tensor — 300 candidate rows of `[x1, y1, x2, y2, confidence, class]`, already deduplicated. Apple's Vision framework does not recognise that as a detector. It returns decoded `VNRecognizedObjectObservation`s, with labels and normalised boxes, only for models carrying Core ML's own NMS metadata — so exporting end-to-end means writing the decode by hand in Swift.

`end2end=False` with `nms=True` therefore trades YOLO26's NMS-free inference for Core ML's built-in NMS layer, and buys a Swift integration of about fifteen lines that cannot get the box maths wrong. Core ML's NMS runs natively rather than in application code, so the cost is small.

**If you would rather keep the end-to-end head**, export with `nms=False, end2end=True` and decode the tensor yourself: filter rows by confidence, scale the four coordinates by the input size, and that is the whole algorithm — no NMS to write, because there is none to do. It is the faster path and the more modern one; it is just more Swift.

Either way the boxes end up as `NormalizedBoundingBox` in the app, with one adjustment for the Vision route: Vision's origin is bottom-left and the app's is top-left, so `y` needs flipping.

In [ ]:
best = YOLO(str(PROJECT / 'runs' / RUN_NAME / 'weights' / 'best.pt'))

exported = best.export(
    format='coreml',
    imgsz=640,
    end2end=False,  # swap YOLO26's NMS-free head for the classic one, so that...
    nms=True,       # ...Core ML's NMS layer makes this a Vision-native detector
    half=True,      # fp16 - half the size, and the Neural Engine runs fp16 anyway
)
print('exported to:', exported)

import shutil
destination = PROJECT / 'HazardDetector.mlpackage'
if destination.exists():
    shutil.rmtree(destination)
shutil.copytree(exported, destination)
print('copied to Drive:', destination)

!du -sh "{destination}"

In [ ]:
# Zip it for download - .mlpackage is a directory, so the browser cannot fetch
# it directly.
!cd "{PROJECT}" && zip -qr HazardDetector.mlpackage.zip HazardDetector.mlpackage

from google.colab import files
files.download(str(PROJECT / 'HazardDetector.mlpackage.zip'))

## 10. Installing it in the app

The app side is already built and waiting. There is exactly one step:

**Put `HazardDetector.mlpackage` in `modules/ar-geospatial/ios/`, commit it, and push.** CI does the rest.

Unzip the download first — it must go in as a directory named `HazardDetector.mlpackage`, not as the `.zip`. It is about 6 MB, which is fine for plain git; no LFS needed.

**Why that directory and not `assets/`.** The `ios/` project is generated by `expo prebuild` and thrown away on every build, so a file added to it by hand does not survive. The module's podspec lists the model as a resource instead, and CocoaPods copies it into the app bundle on every build. The app compiles it to `.mlmodelc` on first launch and caches that, so the one-off compile cost is paid once per install rather than per launch.

**What is already wired**, for reference:

- `HazardDetector.swift` — loads the model, runs it through Vision at 10 Hz, converts boxes from Vision's bottom-left origin to the app's top-left
- `HazardCameraView.swift` — the Hazard Detection screen's camera, AVFoundation only
- `ARGeospatialView.swift` — runs the same detector over ARKit's frames on the AR Navigation screen
- `useHazardDetections` — filters by confidence and by the hazard types left switched on in Settings

Both screens share one detector, so the same pothole is flagged identically whichever tab you are on.

**Class names are the contract.** Vision returns the label string the model was trained with, and the app checks it against `HAZARD_CLASSES`. That is why `CLASSES` at the top of this notebook must not be renamed — a mismatch does not crash, it silently drops every detection.

**Before the model exists**, the app builds and runs fine: the pod resource is simply absent, and the Hazard screen says "Detection unavailable" instead of looking like a camera pointed at flawless pavement.